# Email security of Zurich SMEs

Data: `results/mail_extensive.csv`, produced by the passive DNS scan

```
.venv/bin/python -m sme_scan.scan zh_sme_domains_extensive.csv --out results/mail_extensive.csv
```

joined with the company list (`Category`, `Municipality`) from `zh_sme_domains_extensive.csv`.

## 1. Load and join

The scan output only knows domains; the category lives in the source list. We join on `domain`,
cleaned with **the same `clean_domain` function the scanner used**. If the notebook had its own
cleanup logic, the two could quietly drift apart and rows would stop matching.

`indicator=True` adds a `_merge` column saying which side each row came from: a cheap way to
check that nothing got lost in the join.

In [ ]:
import pandas as pd
from sme_scan.scan import clean_domain

src = pd.read_csv("zh_sme_domains_extensive.csv")
res = pd.read_csv("results/mail_extensive.csv")

src["domain"] = src["Website"].map(clean_domain)

df = src.merge(res, on="domain", how="left", indicator=True)
df["_merge"].value_counts()

Keep only companies we actually have a result for. `unknown` means the DNS lookup failed,
which says nothing about the company, so it must not be counted as "unprotected".

In [ ]:
df = df[(df["_merge"] == "both") & (df["verdict"] != "unknown")].drop(columns="_merge")
print(len(df), "companies with a usable scan result")

## 2. Yes/no columns for each measure

The trick that makes the rest easy: **the mean of a True/False column is the share of True**,
because Python counts `True` as 1 and `False` as 0. So `df["has_dmarc"].mean()` *is* the DMARC adoption rate.

Some columns first need building:
- **Empty CSV cells** load as `NaN`, so "has an SPF record" is `.notna()`.
- **Columns with blanks** (e.g. `spf_too_many_lookups` is blank when there's no SPF at all) load as
  `object` dtype, holding `True`, `False` and `NaN`. `.eq(True)` turns that into a clean boolean
  column where blank counts as False. (Don't use `.astype(bool)` here: `NaN` would become `True`.)

In [ ]:
df["has_spf"] = df["spf_record"].notna()
df["has_dmarc"] = df["verdict"].ne("no DMARC")
df["dmarc_enforced"] = df["verdict"].eq("protected")
df["dkim_found"] = df["dkim_selectors"].notna()
for col in ["dmarc_rua", "mta_sts", "spf_too_many_lookups"]:
    df[col] = df[col].eq(True)

metrics = ["has_spf", "has_dmarc", "dmarc_enforced", "dmarc_rua", "dkim_found"]
(df[metrics].mean() * 100).round(1)

## 3. Who runs the mail, per category?

`pd.crosstab` counts rows for every combination of two columns. `margins=True` adds totals.

In [ ]:
pd.crosstab(df["Category"], df["provider_type"], margins=True, margins_name="total")

Same table as **percent of each row** (`normalize="index"`): read across a row to see how one
category splits between provider types. Rows add up to 100.

In [ ]:
(pd.crosstab(df["Category"], df["provider_type"], normalize="index") * 100).round(0)

## 4. Protection rates by category and by provider type

`groupby(...).mean()` on the boolean columns gives the percentage per group. The `n` column
matters: with ~38 companies per category, **one company is about 2.6 percentage points**, so
differences of a few points between categories are noise, not findings.

In [ ]:
def rates(by):
    """Percent of companies meeting each measure, per group, plus group size n."""
    grouped = df.groupby(by)
    table = (grouped[metrics].mean() * 100).round(0)
    table["n"] = grouped.size()
    return table.sort_values("dmarc_enforced", ascending=False)

rates("Category")

In [ ]:
rates("provider_type")

## 5. Category × provider type

Is the provider effect the same in every category? We keep only the two big provider types,
since the others are too small to split further, and show the **enforcement rate** next to the
**count** in each cell. Always look at both together: 100% of 2 companies is an anecdote.

In [ ]:
main = df[df["provider_type"].isin(["cloud suite", "hoster"])]
pd.pivot_table(
    main,
    index="Category",
    columns="provider_type",
    values="dmarc_enforced",
    aggfunc=["mean", "count"],
).round(2)